In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 7.25 Linear Response and Kubo: How Equilibrium Answers Questions

<!-- This single H1 (one per notebook, "# <number> <Title>") is the page's
     title: it sets the sidebar entry, breadcrumb, browser tab, and search
     result. The branded banner below is generated by the shared `ecp`
     package, so the look of every notebook in the series lives in one place. -->

In [ ]:
from ecp.style import header, use_style

use_style()  # apply the series Matplotlib style
header(
    volume="Volume VII — Quantum Statistical Mechanics",
    number="7.25",
    title="Linear Response and Kubo: How Equilibrium Answers Questions",
    blurb="Every laboratory perturbs: a field applied, a current passed, a light shone. "
    "The deepest practical theorem in statistical mechanics says the results were "
    "never new information — to first order, the response is assembled entirely "
    "from correlations the system already had in equilibrium. We stage that claim "
    "as an actual experiment and watch it hold, measure exactly where linearity "
    "ends, verify fluctuation–dissipation as a pole-by-pole identity, thread a "
    "flux through a ring to learn that a filled band's famous inertness was a "
    "transport theorem all along, and watch interactions move optical weight "
    "around under a conservation law that holds to six digits.",
    difficulty="advanced",
    estimate="205–245 min",
    optional=True,
)

## Notebook overview

The Coda's final notebook asks the question every laboratory asks and this volume never
did: what happens when equilibrium is *disturbed*? Apply a field, pass a current, shine a
light — and the deepest practical theorem in statistical mechanics answers that, to first
order in the disturbance, the results were never new information. The response is
assembled entirely from correlation functions the system already possessed in
equilibrium. Equilibrium knows the answers to questions not yet asked.

A claim that large is not admired here; it is staged. On the chaotic spin chain of
[§7.22](eigenstate-thermalization.ipynb), a weak field is actually switched on at
$t = 0$, the thermal state is evolved *exactly* under the perturbed Hamiltonian, and the
measured response is laid over the Kubo prediction built purely from unperturbed
correlations. They agree at the expected residual — and then the notebook does what
textbooks do not: it measures where linearity *ends*, watching the deviation double as
the field doubles (the $O(\varepsilon^2)$ correction made visible) until the strongest
trace departs from the prediction before the reader's eyes. Linear response is a regime
with edges one can find, and the standing rule is issued: every linear-response number
owes an $\varepsilon$-scan, exactly as every discretization owed an $M$-ladder in
[§7.21](path-integral-monte-carlo.ipynb).

Around that centerpiece, the volume's oldest threads knot. Detailed balance and the
fluctuation–dissipation theorem are verified as the exact pole-by-pole identities they
are — at $10^{-15}$ and $10^{-18}$ — with their famous children named (Johnson–Nyquist
noise thermometry; Einstein's 1905 relation as the first FDT). Causality turns the
retarded $\theta$ into upper-half-plane analyticity and hence Kramers–Kronig: the
complex-analysis arsenal forged in [§7.1](complex-analysis-residues.ipynb) and
[§7.2](complex-analysis-applications.ipynb), finally spent on the physics it was built
for. A flux threaded through the tight-binding ring of
[§7.12](bloch-theorem-band-structure.ipynb) delivers Kohn's criterion with both verdicts:
the filled band's ground energy is flat to machine precision — its famous inertness was a
transport theorem all along, zero Drude weight — while the closed-shell metal's curvature
equals its kinetic energy on the nose. The f-sum rule then shows interactions relocating
optical weight under a conservation law that holds to six digits, and a current
autocorrelation that stays flat when the current is conserved but decays when it is not
closes the loop with the capstone in a single sentence. The gateway ends here; so do the
volume's pages.

> **Conventions (this notebook).** $\hbar = 1$. The retarded susceptibility is
> $\chi_{AB}(t) = -i\,\theta(t)\,\langle[A(t), B(0)]\rangle_{\mathrm{eq}}$ with
> $A(t) = e^{iHt} A e^{-iHt}$; the perturbation is $H \to H + f(t)B$ with $f$ a step.
> The probing field $B = \sum_i \sigma^z_i$ is extensive (a laboratory field couples to
> the whole sample); the meter $A = \sigma^z_{N/2}$ is local. Thermal weights are
> ground-shifted everywhere (the discipline of
> [§7.4](thermal-density-matrix.ipynb)). Spectral evaluations are vectorized weight
> matrices over frequency matrices; the step-response integral is done pole by pole
> analytically, $(e^{i\Omega t} - 1)/i\Omega$, with the $\Omega = 0$ exclusion justified
> in place. Peierls rings are built complex Hermitian (the conjugate bond stated); Kohn
> curvatures use a central second difference with the step stated and the
> finite-difference floor noted. The Kramers–Kronig principal value zeroes the singular
> sample on a symmetric grid. The machinery of
> [§7.22](eigenstate-thermalization.ipynb) (the mixed-field chain) and
> [§7.23](second-quantization.ipynb) (sector logic) is reused, restated with provenance
> so the notebook runs standalone.
>
> **How to read the checks.** Each exercise closes with a `validate` call against an
> independent fact: the spectral susceptibility against a direct commutator evaluation;
> the Kubo prediction against an exact perturbed evolution (and the deviation against
> its $O(\varepsilon^2)$ law); detailed balance and FDT against the identities they are;
> the principal-value integral against the direct $\chi'$; Kohn's curvatures against
> zero and against the kinetic energy; the f-sum against $\langle -T\rangle/N$ at every
> interaction strength; the current correlator against conservation. A ✓ is strong
> evidence; a ✗ is a prompt to *locate the discrepancy*.
>
> **Scope.** Self-energies, Dyson's equation, and diagrammatic perturbation theory
> (Mahan, *Many-Particle Physics*), transport at scale (Boltzmann, Landauer, disorder
> and localization), and Keldysh nonequilibrium theory are beyond the gateway. See Kubo
> 1957; Kubo, Toda & Hashitsume, *Statistical Physics II* (the canon for everything
> below); Kohn, *Phys. Rev.* **133**, A171 (1964); Nyquist 1928 and Johnson 1928;
> Einstein 1905. Cross-reference [§7.1](complex-analysis-residues.ipynb)/
> [§7.2](complex-analysis-applications.ipynb) (the arsenal, spent),
> [§7.9](fermi-gas-zero-temperature.ipynb) (closed shells, returning),
> [§7.10](fermi-gas-finite-temperature.ipynb) (Drude, placed),
> [§7.12](bloch-theorem-band-structure.ipynb) (the inert band's transport meaning),
> [§7.21](path-integral-monte-carlo.ipynb) (the budget discipline, transported),
> [§7.22](eigenstate-thermalization.ipynb) (the handshake),
> [§7.23](second-quantization.ipynb)/[§7.24](greens-functions.ipynb) (the gateway's
> first two rooms).

## Theory in brief

### The claim, and its derivation

Perturb a Hamiltonian by a weak time-dependent term, $H \to H + f(t)B$, and ask how an
observable $A$ moves. Working in the interaction picture and keeping first order in $f$
— three careful steps, standard since Kubo 1957 (Kubo, Toda & Hashitsume, *Statistical
Physics II*, Ch. 4, carries them out in full) — the answer arrives as a convolution
against a memory kernel built *entirely in equilibrium*:

```{math}
:label: eq-kubo
\delta\langle A(t)\rangle = \int_{-\infty}^{t}\! dt'\, \chi_{AB}(t - t')\, f(t'),
\qquad
\chi_{AB}(t) = -i\,\theta(t)\,\big\langle[A(t), B(0)]\big\rangle_{\mathrm{eq}} .
```

Read the three ingredients aloud. The **commutator** is quantum mechanics' fingerprint —
classically this kernel would be a Poisson bracket. The **$\theta(t)$** is causality:
nothing responds before it is pushed. And the **equilibrium average** is the theorem:
the response to tomorrow's perturbation is already written in today's correlations. The
answer predates the question.

### The rendezvous: the claim as an experiment

A claim this large gets staged, not admired. On the chaotic chain of
[§7.22](eigenstate-thermalization.ipynb) ($N = 8$, $\beta = 1$; field
$B = \sum\sigma^z$, meter $A = \sigma^z_{N/2}$), the Kubo prediction for a step
perturbation is assembled from unperturbed spectral data alone,

```{math}
:label: eq-kubo-rendezvous
\delta\langle A(t)\rangle_{\mathrm{Kubo}}
= -i\sum_{m\neq n}(p_m - p_n)\,A_{mn}B_{nm}\,
\frac{e^{i\Omega_{mn}t} - 1}{i\,\Omega_{mn}},
\qquad \Omega_{mn} = E_m - E_n,
```

(the step integral done analytically per pole; the $\Omega = 0$ terms carry
$p_m - p_n = 0$ and are excluded safely) — and then the experiment is actually
performed: diagonalize $H + \varepsilon B$, evolve the thermal state exactly, and
measure $\delta\langle A(t)\rangle/\varepsilon$. Agreement holds at the expected
$O(\varepsilon)$ residual for $\varepsilon = 10^{-3}$, and the *edge of linearity* is
then measured rather than assumed: the deviation doubles as $\varepsilon$ doubles (the
$O(\varepsilon^2)$ correction, demonstrated), and the $\varepsilon = 0.05$ trace departs
visibly. The standing rule: every linear-response number owes an $\varepsilon$-scan,
exactly as every discretization owed an $M$-ladder in
[§7.21](path-integral-monte-carlo.ipynb).

### Detailed balance and FDT, as identities

The correlation spectrum $S_{AA}(\omega)$ is a set of poles at $E_n - E_m$ with weights
$p_m|A_{mn}|^2$, and two celebrated relations are *exact identities* of that list —
taught here at the pole level, not as folklore:

```{math}
:label: eq-fdt
\frac{S(-\omega)}{S(\omega)} = e^{-\beta\omega}
\qquad\text{(detailed balance: it is } p_n/p_m\text{, nothing more)},
\qquad
\chi''(\omega) = \big(1 - e^{-\beta\omega}\big)\, S(\omega),
```

the second being the fluctuation–dissipation theorem, verified below weight by weight.
Dissipation *is* fluctuation. The classical limit hands over the famous children, named
with their years: Johnson and Nyquist, 1928 — the thermal voltage noise
$\overline{V^2} = 4k_BTR\,\Delta f$ across any resistor, FDT's metrological career as
noise thermometry — and Einstein, 1905: $D = \mu k_BT$, diffusion tied to mobility, the
first fluctuation–dissipation relation in physics.

### Causality: the arsenal, spent

$\chi(t < 0) = 0$ is not decoration: it makes $\chi(\omega)$ analytic in the upper
half-plane, and analyticity plus a contour is exactly the machinery the volume forged in
[§7.1](complex-analysis-residues.ipynb) and
[§7.2](complex-analysis-applications.ipynb). The Kramers–Kronig relations follow,

```{math}
:label: eq-kubo-kramers-kronig
\chi'(\omega) = \frac{1}{\pi}\,\mathcal{P}\!\int_{-\infty}^{\infty}
\frac{\chi''(\omega')}{\omega' - \omega}\, d\omega' ,
```

verified numerically below on $\eta$-broadened poles. The physical reading, in one
optics breath: absorption determines refraction — measure $\chi''$ everywhere and
$\chi'$ is owed to you. (And one line of bookkeeping: this *forward* map is benign; it
is inversions that bite, the same geography [§7.24](greens-functions.ipynb) charted.)

### Kohn's flux: the inert band was a transport theorem

How does one ask a *ring* whether it conducts? Kohn's 1964 answer: thread a flux
$\Phi$ through it and watch the ground energy. Minimal coupling puts the flux on the
bonds as Peierls phases (three lines, performed in Exercise 5), and

```{math}
:label: eq-kohn
H(\Phi) = -t\sum_i \big(e^{i\Phi/M} c^\dagger_{i+1}c_i + \text{h.c.}\big) + \cdots,
\qquad
D \;\propto\; \left.\frac{\partial^2 E_0}{\partial\Phi^2}\right|_{\Phi=0},
```

the **Drude weight**: a metal's ground state feels a twist at the boundary, an
insulator's does not — conductivity as ground-state geometry, decades before that phrase
was fashionable (Kohn, *Phys. Rev.* **133**, A171 (1964)). Both verdicts are delivered
below: the *filled* band's curvature is zero to machine precision — the famous inertness
of [§7.12](bloch-theorem-band-structure.ipynb) was always a transport theorem — and the
closed-shell metal obeys $M^2\,\partial^2 E_0/\partial\Phi^2 = \langle -T\rangle$
(derived for the free ring in four lines). One trap is taught first: at half filling the
ring is degenerate at $\Phi = 0$, $E_0(\Phi)$ *kinks*, and a second derivative through a
level crossing lies — the closed-shell rule of
[§7.9](fermi-gas-zero-temperature.ipynb), returning as a transport prerequisite.

### The f-sum: weight moves, never dies

Optical absorption obeys a conservation law. Summing the Drude delta and the regular
(finite-frequency) conductivity gives a total fixed by the kinetic energy alone,

```{math}
:label: eq-fsum
\underbrace{M\,\frac{\partial^2 E_0}{\partial\Phi^2}}_{\text{Drude}}
\;+\;
\underbrace{\frac{2}{M}\sum_{n\neq 0}\frac{|\langle n|\hat\jmath|0\rangle|^2}{\omega_{n0}}}_{\text{regular}}
\;=\; \frac{\langle -T\rangle}{M},
```

so interactions cannot create or destroy optical weight — only relocate it. Verified to
six digits below: the free ring puts *all* weight in the Drude delta (because
$[\hat\jmath, H] = 0$, derived), and switching on a nearest-neighbor $V$ moves part of
it to finite frequency while the sum stays fixed. Light–matter coupling strength is
conserved; interactions only choose where it sits.

### Ballistic or diffusive: the handshake

The same conservation law has a time-domain face: the current autocorrelation

```{math}
:label: eq-handshake
C_{jj}(t) = \big\langle \hat\jmath(t)\,\hat\jmath(0)\big\rangle
\qquad
\begin{cases}
\text{flat forever} & [\hat\jmath, H] = 0 \ \text{(integrable: the Drude delta protected)},\\[2pt]
\text{decays} & \text{interactions on (weight at finite }\omega\text{)}.
\end{cases}
```

One sentence reaches the capstone, with no dependence: response functions live in the
off-diagonal matrix elements that eigenstate thermalization constrains — a chaotic
chain's currents forget, an integrable chain's remember
([§7.22](eigenstate-thermalization.ipynb), saluted). And the Drude picture of
[§7.10](fermi-gas-finite-temperature.ipynb) is placed at last: the phenomenological
$\tau$ of that notebook is what this decay becomes when impurities and phonons supply
it; the ideal chain, having neither, conducts perfectly.

### Onsager, one breath

Time-reversal symmetry ties response coefficients to each other:
$\chi_{AB}(t) = \varepsilon_A\varepsilon_B\,\chi_{BA}(t)$ with $\varepsilon$ the
operators' signatures under time reversal,

```{math}
:label: eq-onsager
\chi_{AB}(t) = \varepsilon_A\,\varepsilon_B\,\chi_{BA}(t),
```

verified below as one numerical identity. Its laboratory face is thermoelectricity:
Seebeck and Peltier coefficients locked together (outward, named).

### Closing the gateway

Alphabet ([§7.23](second-quantization.ipynb)), sentences
([§7.24](greens-functions.ipynb)), and now answers. Beyond the gateway lie
self-energies, diagrams, and transport theory at scale (Mahan, outward); the course's
own road continues in the Epilogue.

## Setup

The stage and nothing else: the mixed-field Ising chain you built in
[§7.22](eigenstate-thermalization.ipynb), restated here as a chaotic sample to probe, the rendezvous
stage assembled once from it ($N = 8$, $\beta = 1$, its eigenbasis, and the meter
$A = \sigma^z_{N/2}$ and field $B = \sum_i\sigma^z_i$ expressed in that basis), and the
plotting colors. Everything this notebook is *about* — the Kubo weights and the retarded
$\chi$, the analytic step response, the exact perturbed evolution it must meet, the
flux-threaded ring and its curvature, the interacting ring's sector Hamiltonian — you
build in the exercise where it is earned.

The Setup below holds this notebook's data and instruments — nothing you
are asked to build. It is collapsed so the building stays yours; expand it
whenever you want the details.

<!-- setup-policy: v2 -->

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from itertools import combinations

from ecp import draw, validate

ACCENT, INK = draw.ACCENT, draw.INK
RED = "#c1121f"

# Conventions: hbar = 1; chi_AB(t) = -i theta(t) <[A(t), B]>_eq; the probing
# field B = sum sigma^z is EXTENSIVE (a laboratory field couples to the whole
# sample), the meter A = sigma^z_{N/2} is local; thermal weights are
# ground-shifted everywhere (the discipline of section 7.4).


# built from scratch in §7.22; restated here as an instrument. The chaotic sample
# this notebook probes, not the thing it teaches: the stage is generic so that the
# response measured on it is generic too.
def H_mixed(N, J=1.0, h=1.05, g=0.5):
    """The mixed-field Ising chain of section 7.22, restated with provenance.

    OBC, bit basis (bit i = site i, 0 = up); (h, g) = (1.05, 0.5) is the
    standard chaotic point. Reused as the rendezvous stage: chaotic, so its
    response is generic.

    Parameters
    ----------
    N : int
        Sites.
    J, h, g : float
        Coupling, transverse and longitudinal fields.

    Returns
    -------
    numpy.ndarray
        The dense Hamiltonian.
    """
    D = 1 << N
    states = np.arange(D)
    z = 1.0 - 2.0 * ((states[:, None] >> np.arange(N)) & 1)
    diag = -J * np.sum(z[:, :-1] * z[:, 1:], axis=1) - g * np.sum(z, axis=1)
    H = np.diag(diag)
    for i in range(N):
        H[states ^ (1 << i), states] += -h
    return H


# data: the rendezvous stage, built once and reused (the cache discipline) — the
# chain, its eigenbasis, and the two operators expressed in it. The thermal weights
# and the Kubo matrices that ride on this stage are assembled in Exercise 1, with
# the machinery that makes them.
N_R = 8
D_R = 1 << N_R
BETA_R = 1.0
states_R = np.arange(D_R)
z_R = 1.0 - 2.0 * ((states_R[:, None] >> np.arange(N_R)) & 1)
H_R = H_mixed(N_R)
B_OP = np.diag(z_R.sum(axis=1))
A_OP = np.diag(z_R[:, N_R // 2])
EV_R, V_R = np.linalg.eigh(H_R)
A_MN = V_R.T @ A_OP @ V_R
B_MN = V_R.T @ B_OP @ V_R

## Exercise 1 — Equilibrium already knows

The Kubo formula derived, and its three ingredients read aloud. Cite {eq}`eq-kubo`.
Setup fixes the stage — the chaotic chain at $N = 8$ and $\beta = 1$, its eigenbasis,
and the operators $A_{mn}$, $B_{mn}$ in it — but the response machinery that rides on
that stage is built here and serves the rest of the notebook. In the eigenbasis the
retarded susceptibility collapses to a sum over pole pairs,
$\chi_{AB}(t) = -i\sum_{mn}W_{mn}e^{i\Omega_{mn}t}$ with weights
$W_{mn} = (p_m - p_n)A_{mn}B_{nm}$ and frequencies $\Omega_{mn} = E_m - E_n$, so every
object below is one vectorized contraction against two matrices — the quadruple loop is
forbidden on the cost grounds of [§7.24](greens-functions.ipynb).

1. Derive linear response in the interaction picture (first order in $f(t)B$, three
   careful steps) to $\chi_{AB}(t) = -i\theta(t)\langle[A(t), B(0)]\rangle_{\rm eq}$.
2. Write `chi_weights(evals, A_mn, B_mn, beta)`, returning $W_{mn}$, $\Omega_{mn}$ and
   the ground-shifted thermal weights $p_m \propto e^{-\beta(E_m - E_0)}$ (the discipline
   of [§7.4](thermal-density-matrix.ipynb)), and `chi_retarded(t, W, Om)` for the
   susceptibility itself. **Write these yourself** — the implementation is the lesson.
3. Apply them to the stage and check the spectral form against a direct commutator
   evaluation $-i\,\mathrm{Tr}\big(\rho\,[e^{iHt}Ae^{-iHt}, B]\big)$ at several times.
4. Read the ingredients (prose): the commutator is quantum's fingerprint, the $\theta$
   is causality, and the equilibrium average is the theorem.
5. State what the notebook will do about it (prose): claims this large get staged as
   experiments — the next exercise switches the field on for real.

In [ ]:
# (solution hidden on the public site)


### Validation 1

In [ ]:
validate.check(
    chi_check_dev < 1e-8,
    "the susceptibility, assembled from equilibrium",
    f"spectral form vs direct commutator: {chi_check_dev:.1e} over three times (two methods, agreeing)",
)

## Exercise 2 — The rendezvous: switch it on (centerpiece)

Kubo's prediction against an actual perturbed evolution — with the edge of linearity
measured. Cite {eq}`eq-kubo-rendezvous`. A step perturbation makes the convolution
{eq}`eq-kubo` a bare time integral $\int_0^t\chi(t')\,dt'$, and in the eigenbasis each
pole integrates in closed form: $\int_0^t e^{i\Omega t'}dt' = (e^{i\Omega t} - 1)/i\Omega$,
undefined at $\Omega = 0$ but carrying weight $p_m - p_n = 0$ there, since thermal
weights depend on energy alone and degenerate levels share $p$ — so dropping those pairs
drops exact zeros, not physics. On the other side stands the experiment itself: nothing
forbids diagonalizing $H + \varepsilon B$ outright and evolving the thermal state under
it with no Trotter and no truncation, which is what a prediction assembled from
*unperturbed* data has to meet.

1. Write `kubo_step(t, W, Om)`, the step response summed pole by pole from the analytic
   integral above, with the $\Omega = 0$ pairs masked out. **Write this one yourself** —
   the implementation is the lesson.
2. Write `exact_step_response(eps, ts, H, B_op, A_op, beta)`: the ground-shifted thermal
   state $\rho(0) = e^{-\beta H}/Z$ built in the unperturbed eigenbasis, then
   $\langle A(t)\rangle$ under $H + \varepsilon B$ by the unitary sandwich in the
   *perturbed* eigenbasis. **Write this one yourself** — the implementation is the lesson.
3. Compute the Kubo step response on the chaotic chain ($N = 8$, $\beta = 1$;
   $B = \sum\sigma^z$, $A = \sigma^z_{N/2}$) from the `chi_weights` matrices you built in
   Exercise 1.
4. Perform the experiment at $\varepsilon = 10^{-3}$ and verify agreement at the
   $O(\varepsilon)$ residual across $t = 0.05$–$3$.
5. Measure the nonlinearity: the deviation from Kubo at $\varepsilon = 0.01, 0.02,
   0.04$, doubling with $\varepsilon$ (the $O(\varepsilon^2)$ law), and the
   $\varepsilon = 0.05$ trace departing visibly.
6. Issue the rule (prose): linear response is a regime with edges one can find — every
   linear-response number owes an $\varepsilon$-scan (the budget discipline of
   [§7.21](path-integral-monte-carlo.ipynb), transported); and say what was just witnessed.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 2

In [ ]:
validate.close(
    resp_1e3,
    KUBO,
    "equilibrium predicted the perturbed future",
    atol=2e-3,
)
validate.check(
    1.7 < ratio_1 < 2.3 and 1.7 < ratio_2 < 2.3,
    "the edge of linearity, measured: deviations double with the field",
    f"ratios {ratio_1:.2f}, {ratio_2:.2f} across eps = 0.01 -> 0.04",
)

## Exercise 3 — Detailed balance and FDT, pole by pole

Two celebrated relations, verified as the identities they are. Cite {eq}`eq-fdt`.

1. Construct $S_{AA}(\omega)$ as poles and weights ($p_m|A_{mn}|^2$ at
   $\omega = E_n - E_m$, the ground-shifted $p$ from your Exercise 1 `chi_weights`)
   and verify detailed balance $S(-\omega)/S(\omega) =
   e^{-\beta\omega}$ across every pole pair above a stated weight floor (the log-ratio
   metric) — and say it plainly: it is the ratio $p_n/p_m$, nothing more.
2. Verify the FDT $\chi'' = (1 - e^{-\beta\omega})S$ weight by weight.
3. Take the classical limit and name the children with their years: Johnson–Nyquist
   1928 ($\overline{V^2} = 4k_BTR\Delta f$, noise thermometry) and Einstein 1905
   ($D = \mu k_BT$, the first FDT).
4. Reflect (prose): the volume's fluctuations — the variances of
   [§7.3](statistical-toolkit.ipynb), the bunching of [§7.7](bose-einstein-fermi-dirac.ipynb) — were never mere noise.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 3

In [ ]:
validate.close(
    [db_dev, fdt_dev],
    [0.0, 0.0],
    "two identities, verified as identities",
    atol=1e-12,
)

## Exercise 4 — Causality cashes nothing it does not own: the arsenal, spent

From $\theta$ to analyticity to Kramers–Kronig, with a numerical check. Cite
{eq}`eq-kubo-kramers-kronig`.

1. Argue $\theta \Rightarrow$ upper-half-plane analyticity $\Rightarrow$ the KK pair
   (the contour of [§7.1](complex-analysis-residues.ipynb)/[§7.2](complex-analysis-applications.ipynb), cited and spent).
2. Verify numerically on $\eta$-broadened poles built from your Exercise 1 weights:
   $\chi'(1.2)$ by direct evaluation against the principal-value integral of $\chi''$
   (`numpy.trapezoid` on a symmetric grid with the singular sample zeroed; why that
   converges, one line).
3. State the physical reading in one optics breath: absorption determines refraction.
4. Note the bookkeeping (one line): the forward KK map is benign; it is inversions that
   bite — the same geography [§7.24](greens-functions.ipynb) charted.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 4

In [ ]:
validate.close(
    chi_re_KK,
    chi_w_direct.real,
    "causality's integral, checked",
    atol=2e-3,
)

## Exercise 5 — Kohn's flux: the inert band was a transport theorem (centerpiece)

Thread a flux; read conductor or insulator off a second derivative. Cite {eq}`eq-kohn`.
The step for the difference below sits between two floors: far above round-off (with
$E_0 \sim 10$ and float64, step $10^{-4}$ puts the curvature floor near $10^{-8}$) and
far below the scale on which $E_0$ actually varies ($\Phi \sim \pi$) — which is why
$10^{-4}$ and not $10^{-8}$ or $10^{-1}$.

1. Derive the Peierls substitution (three lines): minimal coupling puts the flux on the
   bonds as phases $e^{\pm i\Phi/M}$.
2. Write `peierls_ring(M, t_hop, Phi)`, the single-particle ring carrying those phases,
   built complex Hermitian with the conjugate bond written out rather than assumed — a
   silent real cast throws the flux away and every curvature reads zero. **Write this one
   yourself** — the implementation is the lesson.
3. Write `kohn_curvature(E0_of_Phi, step=1e-4)`, the central second difference of $E_0$
   at $\Phi = 0$.
4. Teach the shell trap first: the half-filled ring's $E_0(\Phi)$ kinks through its
   level crossing at $\Phi = 0$ and the second derivative lies; invoke the closed-shell
   rule of [§7.9](fermi-gas-zero-temperature.ipynb) and choose $N_e = 5$.
5. Verify both verdicts: the filled band flat to the finite-difference floor (a
   thousandth of the metal's curvature), and the closed-shell identity
   $M^2\,\partial^2E_0/\partial\Phi^2 = \langle -T\rangle$ (four-line free-ring
   derivation).
6. Read Kohn's criterion (prose): conductivity as ground-state geometry (Kohn 1964
   cited).

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 5

In [ ]:
validate.check(
    abs(curv_filled) < 1e-3 * abs(curv_shell),
    "Kohn's first verdict: the filled band is flat — zero Drude weight",
    f"filled curvature {curv_filled:.1e} vs metal {curv_shell:.1e} (a part in 1e3: zero)",
)
validate.close(
    M_RING**2 * curv_shell,
    kin_shell,
    "and the second: the closed-shell curvature is the kinetic energy",
    atol=1e-4,
)

## Exercise 6 — The f-sum: weight moves, never dies

Interactions relocate optical weight under a conservation law verified to six digits.
Cite {eq}`eq-fsum`. The single-particle ring of Exercise 5 cannot carry an interaction,
so the stage grows: spinless fermions in the $N_e$-particle sector, whose basis is the
ordered occupation tuples ($\binom{M}{N_e}$ of them — the sector logic of
[§7.23](second-quantization.ipynb) in combinatorial dress). Each hop carries both the
Peierls phase and the fermionic sign of the occupied sites jumped over, and the wrap bond
must be included *with its string* or the object is an open chain wearing a ring's name.

1. Write `manybody_ring(M, t_hop, V, Phi, Ne)` for that sector Hamiltonian, complex
   Hermitian, with a nearest-neighbor repulsion $V$ on the diagonal. **Write this one
   yourself** — the implementation is the lesson.
2. Form the current $\hat\jmath = M\,\partial H/\partial\Phi$ at $\Phi = 0$ and derive
   $[\hat\jmath, H] = 0$ for the free ring (why all weight is then ballistic).
3. Verify $V = 0$: Kohn weight $= \langle -T\rangle/M$ exactly (the curvature from your
   Exercise 5 `kohn_curvature`), regular part zero.
4. Verify $V = 1$: Kohn $+$ regular $= \langle -T\rangle/M$ to six digits; draw the
   stacked bars.
5. Extend by assigned verification (expected, not pre-verified): sweep $V = 2, 3$; the
   gate confirms the identity at every $V$.
6. Read the law (prose): interactions choose where optical weight sits, never how much
   exists.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 6

In [ ]:
validate.close(
    [fsum_rows[0.0][0] + fsum_rows[0.0][1], fsum_rows[1.0][0] + fsum_rows[1.0][1]],
    [fsum_rows[0.0][2], fsum_rows[1.0][2]],
    "the f-sum: conserved to six digits",
    rtol=1e-5,
)
validate.check(
    fsum_rows[0.0][1] < 1e-9 and fsum_dev < 1e-5,
    "all weight ballistic at V = 0; the assigned V-sweep confirms the identity at every V",
    f"regular(V=0) = {fsum_rows[0.0][1]:.1e}; max rel dev {fsum_dev:.1e} over V = 0..3",
)

## Exercise 7 — (STUDENT/STRETCH) Currents that remember, currents that forget

The handshake with the capstone, and Drude finally placed. Cite {eq}`eq-handshake`,
{eq}`eq-onsager`.

1. Compute $C_{jj}(t)/C_{jj}(0)$ at $\beta = 2$ on the `manybody_ring` you wrote in
   Exercise 6: flat at $1.0000$ for $V = 0$ (the
   current conserved — derived in Exercise 6) and decaying to $\sim0.91$ at $V = 1$
   (the finite-size plateau and revivals read as physics at $M = 8$, not equilibration
   failure).
2. Make the handshake in one sentence (no dependence): response lives in the
   off-diagonal matrix elements eigenstate thermalization constrains
   ([§7.22](eigenstate-thermalization.ipynb), saluted).
3. Place [§7.10](fermi-gas-finite-temperature.ipynb) at last (prose): the Drude $\tau$ is what this decay becomes when
   impurities and phonons supply it.
4. Verify Onsager reciprocity as one numerical identity
   ($\chi_{AB}(t) = \varepsilon_A\varepsilon_B\chi_{BA}(t)$ on the rendezvous pair, using
   the `chi_weights` and `chi_retarded` you wrote in Exercise 1; the time-reversal signs
   stated).

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 7

In [ ]:
validate.check(
    flat_dev < 1e-10 and c_min < 0.95 and onsager_dev < 1e-12,
    "memory, forgetting, and reciprocity",
    f"V=0 flat to {flat_dev:.1e}; V=1 dips to {c_min:.3f}; Onsager {onsager_dev:.1e}",
)

## Exercise 8 — (Synthesis) The gateway, closed

No new computation: what the Coda built, and where it leaves the reader.

Three notebooks ago the Coda opened with an alphabet; this one ends with equilibrium
answering questions it was never asked. The claim at the center — that a system's
response to tomorrow's perturbation is written in today's correlations — was not admired
but staged: a field switched on, a thermal state evolved exactly, and the prediction
assembled from unperturbed data holding to the expected residual, with the edge of
linearity located and measured like any other budget. Around it, the volume's threads
knotted. Fluctuation and dissipation revealed as one bookkeeping identity at fifteen
digits; the opening arsenal's contours finally buying nothing less than Kramers–Kronig;
a filled band's famous inertness unmasked as a transport theorem; optical weight obeying
a conservation law to six digits while interactions merely relocate it; and a conserved
current keeping the memory that a chaotic one forgets — the capstone saluted in one
sentence, from the other side of the gateway.

It is worth sitting for a moment with what Kubo's theorem says about equilibrium. We
spent a volume treating it as the state where nothing happens; it turns out to be the
state where everything that *could* happen is already priced in. Equilibrium is not the
absence of dynamics — it is dynamics, fully rehearsed, waiting for a cue.

The Many-Body Gateway now stands: occupation lists with operators
([§7.23](second-quantization.ipynb)), propagators with sum rules
([§7.24](greens-functions.ipynb)), responses with budgets. And the gateway opens onto
the course's own country: [Volume VIII](../08-electronic-structure/index.md) begins on
the next page with the many-electron problem itself, and everything built here — the
operators, the spectral functions, the response formalism — becomes its working
language, through Hartree–Fock, density-functional theory, real band structures, the
Hubbard model, GW, and superconductivity. After it, the Epilogue closes the course.

## Notebook summary

The Coda's third notebook, and the volume's final page: equilibrium, asked.

- **Kubo** {eq}`eq-kubo`: derived in three careful steps; the commutator (quantum), the
  $\theta$ (causality), the equilibrium average (the theorem); the spectral form checked
  against a direct commutator trace at $10^{-15}$ (gated).
- **The rendezvous** {eq}`eq-kubo-rendezvous`: the step response from equilibrium data alone
  against an exact perturbed evolution — agreement at the $O(\varepsilon)$ residual for
  $\varepsilon = 10^{-3}$ (gated); the nonlinearity measured, deviations doubling with
  $\varepsilon$ (gated) and the $\varepsilon = 0.05$ trace departing. Standing rule:
  every linear-response number owes an $\varepsilon$-scan.
- **Detailed balance and FDT** {eq}`eq-fdt`: the weight-ratio identity at
  $4\times10^{-15}$ over 57,308 pole pairs and the FDT weight-by-weight at $10^{-18}$
  (both gated); Johnson–Nyquist 1928 and Einstein 1905 named as the classical children.
- **Kramers–Kronig** {eq}`eq-kubo-kramers-kronig`: causality $\Rightarrow$ analyticity
  $\Rightarrow$ the pair; the principal-value integral meets the direct $\chi'$ at the
  quadrature floor (gated); absorption determines refraction.
- **Kohn** {eq}`eq-kohn`: Peierls phases in three lines; the shell trap taught (the
  half-filled kink's lying second derivative) before $N_e = 5$; the filled band flat to the
  finite-difference floor — its curvature a thousandth of the metal's, zero Drude weight,
  the inert band's transport meaning — and the closed-shell
  $M^2\partial^2E_0/\partial\Phi^2 = \langle -T\rangle$ to the finite-difference floor
  (both gated).
- **The f-sum** {eq}`eq-fsum`: Drude $+$ regular $= \langle -T\rangle/M$ to six digits
  at $V = 0$ and $1$ (gated), with the assigned $V = 2, 3$ sweep confirming the
  identity at every interaction (gated): weight moves, never dies.
- **The handshake** {eq}`eq-handshake`, {eq}`eq-onsager`: $C_{jj}$ flat under
  conservation and decaying under $V$ (gated; plateau-is-physics noted); the one-
  sentence salute to [§7.22](eigenstate-thermalization.ipynb); the Drude $\tau$ of [§7.10](fermi-gas-finite-temperature.ipynb) placed; Onsager
  reciprocity verified as one identity (gated).

Standing rules issued here: every linear-response number owes an $\varepsilon$-scan;
build Peierls matrices complex Hermitian; never differentiate through a level crossing;
and read finite-size plateaus as physics before blaming equilibration.

## Outlook

- **Self-energies, Dyson's equation, diagrams**: Dyson and the self-energy arrive in
  [§8.14](../08-electronic-structure/quasiparticles-gw.ipynb); the diagrammatic
  machinery at scale stays beyond the gateway (Mahan; outward).
- **Transport at scale**: Boltzmann equations, Landauer conductance, disorder and
  localization (outward, named).
- **Nonequilibrium beyond linear**: Keldysh, in one honest name (outward).
- **Noise thermometry and metrology**: the Johnson–Nyquist career (outward).
- **The Epilogue**: the course's own last destination.
- Cross-reference: [§7.1](complex-analysis-residues.ipynb)/
  [§7.2](complex-analysis-applications.ipynb) (the arsenal, spent),
  [§7.9](fermi-gas-zero-temperature.ipynb) (shells, returning),
  [§7.10](fermi-gas-finite-temperature.ipynb) (Drude, placed),
  [§7.12](bloch-theorem-band-structure.ipynb) (the theorem it always was),
  [§7.21](path-integral-monte-carlo.ipynb) (budgets, transported),
  [§7.22](eigenstate-thermalization.ipynb) (the salute),
  [§7.23](second-quantization.ipynb)/[§7.24](greens-functions.ipynb) (the gateway's
  first two rooms).

In [ ]:
from ecp.style import footer

footer()